# PHASE 3: LLM EXTRACTION & PERFORMANCE EVALUATION

## 1. Objective
- Initialize and configure a Large Language Model (LLM - Gemini 3.1 Flash Lite) in Deterministic mode (Temperature = 0.0) to ensure Reproducibility.
- **Part 1 - Evaluation:** Use the LLM to extract from a set of 100 records and compare against the Ground Truth using mathematical coefficients to demonstrate automation capability.
- **Part 2 - Inference:** Apply the LLM to automatically label the remaining 150 raw data records in the Pipeline.

In [4]:
%pip install google-genai pandas numpy scikit-learn python-dotenv tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
import json
import time
import os
import numpy as np
from sklearn.metrics import cohen_kappa_score
from dotenv import load_dotenv
from google import genai
from google.genai import types
from tqdm.auto import tqdm  

# Load API key from environment file
load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")
if not API_KEY:
    raise ValueError("Error: GEMINI_API_KEY not found in the .env file.")

client = genai.Client(api_key=API_KEY)
print("Connected to Google Gemini API successfully.")

Connected to Google Gemini API successfully.


In [6]:
GT_FILE_PATH = "../data/02_annotation/GroundTruth_100.csv"
df_gt = pd.read_csv(GT_FILE_PATH)
print(f"GroundTruth dataset loaded: {df_gt.shape[0]} rows.")

# Define label vocabularies
DOMAINS = ["Software Development", "Data & AI", "Design & UX", "Management & Analysis", "Infrastructure", "Testing & Quality", "Others"]
LANGUAGES = ["English", "Japanese", "Korean", "French", "None"]
SKILLS_LIST = [
    "Angular", "API", "API Testing", "Automation Test", "AWS", "Azure", "Big Data", "Blockchain", "Business Analysis",
    "C/C++", "C#", "CI/CD", "Computer Vision", "Data Analysis", "Data Engineering", "Data Visualization", "Data Warehousing",
    "Deep Learning", "Design", "Django", "Docker", "Embedded / IoT", "FastAPI", "Firewall / VPN", "Flask", "Flutter",
    "Frontend Basics", "Game Development", "GCP", "GenAI / LLM", "Git", "Golang", "GraphQL", "gRPC", "Java", "JavaScript",
    "Kubernetes", "Langchain", "Laravel", "Linux", "Machine Learning", "Manual Test", "Microservices", "MLOps", "Mobile Apps",
    "Monitoring", "MySQL / PostgreSQL", "Networking", "NextJS", "NLP", "NodeJS", "NoSQL", "PHP", "Python", "RAG", "React Native",
    "ReactJS", "Redis", "Ruby", "Rust", "SAP", "Scala", "Scrum / Agile", "Security", "Software Architecture", "Spring Boot",
    "SQL", "System Design", "Terraform", "Transformer", "TypeScript", "Unit Test", "Vector Database", "VueJS", "Web Server / Proxy",
    "WebSocket", ".NET Ecosystem"
]

GroundTruth dataset loaded: 100 rows.


In [7]:
def call_gemini_extraction(job_title, jd_text):
    prompt = f"""You are an Expert IT Recruiter in Vietnam.
Your task is to read the Job Title and Job Description (JD), then extract the following information in EXACT JSON format. DO NOT provide any additional explanation.

[EXTRACTION CONSTITUTION - STRICT COMPLIANCE REQUIRED]
1. "Job_Domain": Select EXACTLY 1 value from the following list: {DOMAINS}. (Absolutely do not create new domains. If unclear, select "Others").
2. "Min_years_of_exp": Extract the MINIMUM years of experience required. (Return as a float, e.g., 0.0 for fresher/no experience, 2.0 for 2 years). Return null if the JD does not mention it.
3. "Language_Requirement": Select EXACTLY 1 value from: {LANGUAGES}. 
   - PRIORITY RULE: If the JD requires multiple languages (e.g., both English and Korean), YOU MUST prioritize the more niche/specific language (Japanese/Korean/French).
   - If no language is explicitly mentioned, select "None".
4. "Skills": Extract the list of required skills.
   - STRICT FILTER: YOU ARE ONLY ALLOWED to select exact keywords from the following Whitelist: {SKILLS_LIST}. NEVER invent new terms.
   - SKILL ABSORPTION: If the JD requires specific tools not in the Whitelist (e.g., Oracle, Kafka), map them to their corresponding root labels (e.g., SQL, Microservices) if conceptually accurate.
   - STRICT DEFINITIONS (No label inflation):
     + "Business Analysis": ONLY select when the role requires requirements elicitation or writing project documents (BRD/SRS).
     + "System Design": ONLY select for designing overall architecture or distributed systems.
     + "Monitoring": ONLY select for operating automated monitoring systems (Metrics/Logs like ELK, Grafana).
     + "Data Analysis": ONLY select when using data to extract business insights.
     + "Manual Test": ONLY select for QA/QC roles designing software test cases.

[INPUT DATA]
- Job Title: {job_title}
- JD: {jd_text}

[REQUIRED OUTPUT JSON FORMAT]
{{
    "Job_Domain": "string",
    "Min_years_of_exp": float or null,
    "Language_Requirement": "string",
    "Skills": ["skill1", "skill2"]
}}
"""
    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model='gemini-3.1-flash-lite',
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=0.0, 
                )
            )
            return json.loads(response.text)
        except Exception as e:
            error_msg = str(e)
            print(f"\nAPI error on current row (Attempt {attempt + 1}/3): {error_msg}")
            
            if attempt < 2:
                print("Server is overloaded. Waiting 15 seconds before retrying...")
                time.sleep(15)
            else:
                print("All 3 attempts failed. Skipping this row.")
                return {"Job_Domain": None, "Min_years_of_exp": None, "Language_Requirement": None, "Skills": []}

In [8]:
print("Starting AI extraction on the 100-row Ground Truth dataset...")

ai_domains, ai_exps, ai_langs, ai_skills = [], [], [], []

for index, row in tqdm(df_gt.iterrows(), total=df_gt.shape[0], desc="LLM Processing"):
    title = row['Job_Title']
    jd = row['JD & Requirements']
    extracted = call_gemini_extraction(title, jd)
    ai_domains.append(extracted.get("Job_Domain"))
    ai_exps.append(extracted.get("Min_years_of_exp"))
    ai_langs.append(extracted.get("Language_Requirement"))
    ai_skills.append(", ".join(extracted.get("Skills", [])))

    time.sleep(4)  

df_gt['AI_Job_Domain'] = ai_domains
df_gt['AI_Min_exp'] = ai_exps
df_gt['AI_Language'] = ai_langs
df_gt['AI_Skills'] = ai_skills  

# Save the comparison file
df_gt.to_csv("../data/03_interim/AI_100.csv", index=False)
print("Comparison file saved to data/03_interim/AI_100.csv.")

Starting AI extraction on the 100-row Ground Truth dataset...


LLM Processing: 100%|██████████| 100/100 [08:58<00:00,  5.38s/it]

Comparison file saved to data/03_interim/AI_100.csv.


In [9]:
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score

FILE_PATH = "../data/03_interim/AI_100.csv"
df_eval = pd.read_csv(FILE_PATH)

print("LLM EXTRACTION CAPABILITY EVALUATION SCORECARD (AI vs GROUNDTRUTH): ")

def clean_to_set(s):
    if pd.isna(s) or str(s).strip() == '': return set()
    return set([x.strip().lower() for x in str(s).split(',') if x.strip()])

def calc_jaccard(s1, s2):
    set1, set2 = clean_to_set(s1), clean_to_set(s2)
    if not set1 and not set2: return 1.0
    if not set1 or not set2: return 0.0
    return len(set1.intersection(set2)) / len(set1.union(set2))

def normalize_text(series):
    return series.fillna('None').astype(str).str.lower().str.strip()

# Calculate Cohen's Kappa for categorical fields
kappa_domain = cohen_kappa_score(normalize_text(df_eval['Job Domain']), normalize_text(df_eval['AI_Job_Domain']))
kappa_exp = cohen_kappa_score(normalize_text(df_eval['Min years of exp']), normalize_text(df_eval['AI_Min_exp']))
kappa_lang = cohen_kappa_score(normalize_text(df_eval['Language Requirement']), normalize_text(df_eval['AI_Language']))

jaccard_scores = [calc_jaccard(df_eval['Skills'].iloc[i], df_eval['AI_Skills'].iloc[i]) for i in range(len(df_eval))]
mean_jaccard = np.mean(jaccard_scores)

print(f"1. Domain Agreement (Cohen's Kappa):          {kappa_domain:.4f}")
print(f"2. Experience Agreement (Cohen's Kappa):      {kappa_exp:.4f}")
print(f"3. Language Agreement (Cohen's Kappa):        {kappa_lang:.4f}")
print(f"4. Skills Extraction Performance (Jaccard):   {mean_jaccard * 100:.2f}%")

LLM EXTRACTION CAPABILITY EVALUATION SCORECARD (AI vs GROUNDTRUTH): 
1. Domain Agreement (Cohen's Kappa):          0.9371
2. Experience Agreement (Cohen's Kappa):      0.8366
3. Language Agreement (Cohen's Kappa):        0.8402
4. Skills Extraction Performance (Jaccard):   76.35%


The evaluation metrics (Kappa > 0.83 and Jaccard > 76%) demonstrate that the Prompt Engineering ruleset performs extremely well, effectively controlling AI hallucinations and achieving a very high degree of alignment with human labeling judgment. At this level of accuracy, the system has met the threshold for reliability. We will use this same Prompt structure to perform automated extraction (Inference) on the remaining 150 raw data records in the project.

In [10]:
# Load 150 unlabeled rows from Phase 2
UNLABELED_PATH = "../data/03_interim/150_jobs_for_pipeline.csv"
df_unlabeled = pd.read_csv(UNLABELED_PATH)

print(f"Starting AI extraction pipeline for {df_unlabeled.shape[0]} records...")

final_domains, final_exps, final_langs, final_skills = [], [], [], []

# Run extraction loop (leveraging call_gemini_extraction with built-in 503 error handling)
for index, row in tqdm(df_unlabeled.iterrows(), total=df_unlabeled.shape[0], desc="LLM Inference 150 Rows"):
    title = row.get('Job_Title', row.get('Job Title', ''))
    jd = row.get('JD & Requirements', row.get('JD', ''))
    
    extracted = call_gemini_extraction(title, jd)
    
    final_domains.append(extracted.get("Job_Domain"))
    final_exps.append(extracted.get("Min_years_of_exp"))
    final_langs.append(extracted.get("Language_Requirement"))
    final_skills.append(", ".join(extracted.get("Skills", [])))
    
    time.sleep(5) 

df_unlabeled['Job_Domain'] = final_domains
df_unlabeled['Min_years_of_exp'] = final_exps
df_unlabeled['Language_Requirement'] = final_langs
df_unlabeled['Skills'] = final_skills

# Export the final output file for Phase 3
FINAL_OUTPUT_PATH = "../data/03_interim/150_llm_extracted.csv"
df_unlabeled.to_csv(FINAL_OUTPUT_PATH, index=False)

print(f"\nAll data has been processed by AI and saved to: {FINAL_OUTPUT_PATH}")

Starting AI extraction pipeline for 150 records...


LLM Inference 150 Rows:   0%|          | 0/150 [00:00<?, ?it/s]

LLM Inference 150 Rows: 100%|██████████| 150/150 [18:30<00:00,  7.40s/it]


All data has been processed by AI and saved to: ../data/03_interim/150_llm_extracted.csv


In [12]:
print("Merging and cleaning unused columns...")

df_100_gt = pd.read_csv("../data/02_annotation/GroundTruth_100.csv")
df_150_ai = pd.read_csv("../data/03_interim/150_llm_extracted.csv")

# Synchronize column names (align AI output file to match Ground Truth file)
df_150_ai.rename(columns={
    'Job_Domain': 'Job Domain',
    'Min_years_of_exp': 'Min years of exp',
    'Language_Requirement': 'Language Requirement'
}, inplace=True)

# Concatenate both DataFrames into one complete dataset (100 + 150 = 250)
df_final = pd.concat([df_100_gt, df_150_ai], ignore_index=True)

# Drop columns no longer needed for analysis
columns_to_drop = ['List Skills']
df_final.drop(columns=[col for col in columns_to_drop if col in df_final.columns], inplace=True)

FINAL_DATASET_PATH = "../data/03_interim/250_merged_raw.csv"
df_final.to_csv(FINAL_DATASET_PATH, index=False)

print(f"- Total rows: {df_final.shape[0]}")
print(f"- Total columns: {df_final.shape[1]}")
print(f"- Final preprocessed dataset saved to: {FINAL_DATASET_PATH}")

display(df_final.sample(5))

Merging and cleaning unused columns...
- Total rows: 250
- Total columns: 10
- Final preprocessed dataset saved to: ../data/03_interim/250_merged_raw.csv


,Job_Title,Job Domain,Company,Location,Min years of exp,Language Requirement,Skills,Salary_Raw,Link,JD & Requirements
66,Backend Engineer,Software Development,Smilegate Vietnam,"Quận Bình Thạnh, Hồ Chí Minh",3.0,Korean,"Python, Java, Microservices, API, gRPC, SQL, M...",You'll love it,https://itviec.com/it-jobs/backend-engineer-ai...,Job description\n \n Backend Engineering:\n \n...
95,Tester / QA QC Engineer,Testing & Quality,Ohmyhotel & Co Group,"Quận Bình Thạnh, Hồ Chí Minh",3.0,English,"API, API Testing, Automation Test, CI/CD, Dock...",Negotiable,https://topdev.vn/detail-jobs/tester-qa-qc-eng...,Your role & responsibilities\n \n Quality Engi...
178,Mobile Application Engineer (Android/iOS/Swift),Software Development,OCB,"The Hallmark Building, 15 Tran Bach Dang Stree...",5.0,NaN,"API, Automation Test, CI/CD, Git, Mobile Apps,...",You'll love it,https://itviec.com/it-jobs/senior-mobile-appli...,Job description\r\nWe are on a journey to mode...
154,Process Quality Assurance,Testing & Quality,FPT Software,"Tòa nhà FPT Cầu Giấy, phố Duy Tân, phường Cầu ...",NaN,Japanese,NaN,You'll love it,https://itviec.com/it-jobs/process-quality-ass...,Job description\r\nABOUT FPT SOFTWARE\r\n\r\nF...
108,Automation Tester (QA QC),Testing & Quality,ABBANK,"36 Hoàng Cầu, O Cho Dua, Ha Noi",2.0,NaN,"Automation Test, API, Manual Test, Scrum / Agi...","1,000 - 2,000 USD",https://itviec.com/it-jobs/automation-tester-q...,Job description\r\nXây dựng dụng cụ cho automa...
